<a href="https://colab.research.google.com/github/lkshbt/lassi/blob/main/Streamlit_Frontend_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Phases 3 & 7: Streamlit UI and Interactive Dashboard
import streamlit as st
import requests
import pandas as pd
import plotly.express as px
import time

BACKEND_URL = "http://backend:8000"

st.set_page_config(page_title="AquaBioID Platform", page_icon="🧬", layout="wide")

# Phase 3: Scientific Orange Theme and DNA Animation CSS
st.markdown("""
<style>
    :root {
        --primary: #FF6A00;
        --bg: #0F1115;
    }
    .stApp {
        background-color: var(--bg);
        color: #F0F2F6;
    }
    h1, h2, h3 { color: #FF6A00; }

    /* Simple CSS representation of a DNA vibe / landing header */
    .dna-header {
        background: linear-gradient(90deg, rgba(255,106,0,1) 0%, rgba(200,50,0,1) 100%);
        padding: 20px;
        border-radius: 10px;
        text-align: center;
        margin-bottom: 30px;
        box-shadow: 0 4px 15px rgba(255,106,0,0.3);
    }
    .stButton>button {
        background-color: #FF6A00;
        color: white;
        border: none;
    }
    .stButton>button:hover {
        background-color: #FF8800;
        color: white;
    }
</style>
""", unsafe_allow_html=True)

st.markdown('<div class="dna-header"><h2>🧬 AquaBioID - eDNA Analysis Platform</h2><p>Production-grade biodiversity assessment</p></div>', unsafe_allow_html=True)

tab1, tab2, tab3 = st.tabs(["📤 Upload & Process", "📊 Dashboard", "📄 Reports"])

with tab1:
    st.subheader("Submit Sequencing Data (FASTA/FASTQ)")
    uploaded_file = st.file_uploader("Drag and drop your raw read files here", type=["fastq", "fq", "fasta", "fa"])

    if st.button("Start Pipeline"):
        if uploaded_file is not None:
            with st.spinner('Uploading and initiating pipeline...'):
                files = {"file": (uploaded_file.name, uploaded_file.getvalue())}
                response = requests.post(f"{BACKEND_URL}/upload/", files=files)
                if response.status_code == 200:
                    sample_id = response.json()["sample_id"]
                    st.success(f"Upload successful! Job ID: {sample_id}")
                    st.session_state['current_sample'] = sample_id
                else:
                    st.error("Upload failed.")
        else:
            st.warning("Please upload a file first.")

    st.divider()
    st.subheader("Job Tracker")
    job_id = st.text_input("Enter Job ID to track status", value=st.session_state.get('current_sample', ''))
    if st.button("Check Status"):
        if job_id:
            res = requests.get(f"{BACKEND_URL}/status/{job_id}")
            if res.status_code == 200:
                status = res.json()['status']
                if status == "Processing":
                    st.info(f"⚙️ Status: {status} (Quality Control -> Trimming -> VSEARCH -> BLAST)")
                elif status == "Completed":
                    st.success(f"✅ Status: {status}")
                else:
                    st.warning(f"Status: {status}")
            else:
                st.error("Job ID not found.")

with tab2:
    st.subheader("Biodiversity Dashboard")
    dash_job = st.text_input("Enter Job ID to load dashboard", value=st.session_state.get('current_sample', ''), key="dash_input")

    if st.button("Load Analytics"):
        if dash_job:
            with st.spinner("Fetching taxonomy and metrics..."):
                tax_res = requests.get(f"{BACKEND_URL}/results/{dash_job}/taxonomy")
                div_res = requests.get(f"{BACKEND_URL}/results/{dash_job}/diversity")

                if tax_res.status_code == 200 and div_res.status_code == 200:
                    tax_data = tax_res.json()
                    div_data = div_res.json()

                    if len(tax_data) > 0:
                        # Metrics Cards
                        col1, col2, col3, col4 = st.columns(4)
                        col1.metric("Shannon Index", f"{div_data['shannon']:.3f}")
                        col2.metric("Simpson Index", f"{div_data['simpson']:.3f}")
                        col3.metric("Chao1 Richness", f"{div_data['chao1']:.3f}")
                        col4.metric("Observed Species", div_data['observed'])

                        df = pd.DataFrame(tax_data)

                        st.divider()
                        col_chart1, col_chart2 = st.columns(2)

                        with col_chart1:
                            # Taxonomy Tree / Sunburst
                            st.write("### Taxonomic Distribution")
                            fig = px.sunburst(df, path=['phylum', 'class_name', 'genus', 'species'], values='abundance',
                                              color_discrete_sequence=px.colors.sequential.Oranges)
                            st.plotly_chart(fig, use_container_width=True)

                        with col_chart2:
                            # Species Abundance Bar Chart
                            st.write("### Top Species Abundance")
                            top_species = df.groupby('species')['abundance'].sum().reset_index().sort_values('abundance', ascending=False).head(15)
                            fig2 = px.bar(top_species, x='abundance', y='species', orientation='h', color='abundance',
                                          color_continuous_scale='Oranges')
                            fig2.update_layout(yaxis={'categoryorder':'total ascending'})
                            st.plotly_chart(fig2, use_container_width=True)

                        # Raw Data Table
                        st.write("### OTU/ASV Table")
                        st.dataframe(df[['otu_id', 'genus', 'species', 'abundance', 'confidence']], use_container_width=True)
                    else:
                        st.info("Pipeline is still processing or no species were matched.")
                else:
                    st.error("Failed to fetch data or Job ID invalid.")

with tab3:
    st.subheader("Generate & Download Reports")
    rep_job = st.text_input("Enter Job ID to download report", value=st.session_state.get('current_sample', ''), key="rep_input")

    if st.button("Generate PDF Report"):
        if rep_job:
            st.markdown(f"**[Click here to download PDF Report]({BACKEND_URL}/download/report/{rep_job})**")

            # Allow CSV downloads directly from Streamlit frontend
            st.write("### Export Data")
            tax_res = requests.get(f"{BACKEND_URL}/results/{rep_job}/taxonomy")
            if tax_res.status_code == 200:
                df_export = pd.DataFrame(tax_res.json())
                if not df_export.empty:
                    csv = df_export.to_csv(index=False).encode('utf-8')
                    st.download_button(
                        label="Download Taxonomy Table (CSV)",
                        data=csv,
                        file_name=f'aquabioid_taxonomy_{rep_job}.csv',
                        mime='text/csv',
                    )